<a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_10/05_AppObjectDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🛠️ 1. INSTALLATION
!pip install -q ultralytics gradio

In [ ]:
# 🔄 2. IMPORTS
import gradio as gr
from PIL import Image
import torch
from ultralytics import YOLO
import tempfile

In [ ]:
# 🧠 3. LOAD YOLOE MODEL
model = YOLO('yoloe-11s-seg.pt')  # you can use 'yoloe-11s-seg.pt', '11m-seg.pt', or '11l-seg.pt'

In [ ]:
# 🔍 4. INFERENCE FUNCTION
def detect_objects(image: Image.Image, object_names: str):
    if not object_names.strip():
        return image, "Please enter object names."

    # Split the object names by comma
    classes = [obj.strip().lower() for obj in object_names.split(',') if obj.strip()]

    # Run detection with YOLOE model
    model.set_classes(classes, model.get_text_pe(classes))
    results = model.predict(image)

    # Get the annotated image from results
    annotated_img = results[0].plot()  # plots bounding boxes

    return Image.fromarray(annotated_img), f"Detected: {', '.join(classes)}"

In [ ]:
# 🎛️ 5. GRADIO INTERFACE
with gr.Blocks() as demo:
    gr.Markdown("## 🧠 YOLO-World Objekterkennung")
    with gr.Row():
        image_input = gr.Image(type="pil", label="Bild hochladen")
        text_input = gr.Textbox(label="Zu erkennende Objekte (kommagetrennt)", placeholder="z. B. cat, dog, person")
    detect_button = gr.Button("Objekte erkennen")
    output_image = gr.Image(label="Ergebnisbild")
    status = gr.Textbox(label="Status")

    detect_button.click(fn=detect_objects, inputs=[image_input, text_input], outputs=[output_image, status])

# 🚀 6. LAUNCH APP
demo.launch()